# SPM 405: Machine Learning — Lecture 5 notebook
## Non-linear models and trees

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Fit a building's daily electricity demand with **polynomials, step functions and splines**, and choose the number of knots by CV.
2. Add outdoor temperature in a **generalised additive model**.
3. Classify machine faults from six sensors with a **decision tree**, **bagging**, a **random forest** and **boosting**.

Reading: ISLP 7.1–7.4, 7.7 and 8.1–8.2. Run each cell with **Shift + Enter**, from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, KBinsDiscretizer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers as the slides

## Part 1: A building's electricity demand over the day

120 smart-meter readings at random hours. Demand has a morning and an evening peak, and rises with outdoor temperature (air conditioning).

In [ ]:
def demand(h, temp):
    return 20 + 25 * np.exp(-0.5 * ((h - 9) / 2)**2) + 35 * np.exp(-0.5 * ((h - 19.5) / 1.8)**2) + 0.8 * (temp - 22)

n = 120
h = rng.uniform(0, 24, n)
temp = rng.normal(24, 4, n)                        # outdoor temperature (deg C)
y = demand(h, temp) + rng.normal(0, 4, n)          # noise variance 16 kW^2

cv = KFold(10, shuffle=True, random_state=0)
def cv_mse(model, X):
    return -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error").mean()

H = h[:, None]
models = {
    "straight line": LinearRegression(),
    "polynomial, degree 4": make_pipeline(PolynomialFeatures(4), LinearRegression()),
    "6 steps": make_pipeline(KBinsDiscretizer(6, encode="onehot", strategy="uniform"), LinearRegression()),
    "spline, 9 knots": make_pipeline(SplineTransformer(n_knots=9, degree=3), LinearRegression()),
}
for name, m in models.items():
    print(f"{name:22s} 10-fold CV error {cv_mse(m, H):6.1f}")

g = np.linspace(0, 24, 300)
plt.plot(h, y, ".", color="gray")
for name, m in models.items():
    plt.plot(g, m.fit(H, y).predict(g[:, None]), label=name)
plt.plot(g, demand(g, 24), "k--", label="truth")
plt.xlabel("hour of day"); plt.ylabel("demand (kW)"); plt.legend(); plt.show()

In [ ]:
for k in [4, 5, 6, 9, 12, 25]:
    m = make_pipeline(SplineTransformer(n_knots=k, degree=3), LinearRegression())
    print(f"{k:2d} knots: CV error {cv_mse(m, H):6.1f}")

### A GAM: a spline for the hour, plus a straight line for temperature

In [ ]:
X2 = np.c_[h, temp]
gam = make_pipeline(
    ColumnTransformer([("hour", SplineTransformer(n_knots=9, degree=3), [0]), ("temp", "passthrough", [1])]),
    LinearRegression())
print("hour only (spline):      CV error", round(cv_mse(models["spline, 9 knots"], H), 1))
print("hour spline + temperature: CV error", round(cv_mse(gam, X2), 1))
gam.fit(X2, y)
print("temperature coefficient:", round(gam[-1].coef_[-1], 2), "kW per °C  (true 0.8)")

**Questions**
1. Why does the degree-4 polynomial fail, while a spline with the same kind of cubic pieces succeeds?
2. 5 knots does worse than 4 or 6. The knots are evenly spaced over the day: plot the 5-knot fit and see how it handles the two peaks. What does this tell you about choosing knots by hand?
3. The GAM's CV error (18.8) is close to 16. What is 16, and why can no model beat it?

## Part 2: Machine faults from six sensors

Faults follow hidden rules: high vibration, **or** hot **and** high current, **or** old **and** noisy. 5% of labels are random, so the best possible error is about 0.05.

In [ ]:
names = ["vibration", "temperature", "current", "noise", "hours", "humidity"]
def plant(m):
    vib = rng.gamma(4, 0.8, m)                     # mm/s
    temp = rng.normal(55, 10, m)                   # bearing temperature (deg C)
    cur = rng.normal(10, 2, m)                     # motor current (A)
    noise_db = rng.normal(70, 5, m)                # acoustic level (dB)
    hours = rng.uniform(0, 20000, m)               # running hours
    hum = rng.uniform(20, 90, m)                   # humidity (%), plays no part
    risk = (vib > 4.5) | ((temp > 65) & (cur > 11)) | ((hours > 15000) & (noise_db > 75))
    flip = rng.uniform(size=m) < 0.05              # 5% of labels are wrong
    return np.c_[vib, temp, cur, noise_db, hours, hum], (risk ^ flip).astype(int)

X_train, y_train = plant(500)
X_test, y_test = plant(5000)
print("fraction faulty: train", y_train.mean().round(3), " test", y_test.mean().round(3))

def report(model, name, standardise=False):
    A, B = X_train, X_test
    if standardise:
        sc = StandardScaler().fit(X_train); A, B = sc.transform(X_train), sc.transform(X_test)
    model.fit(A, y_train)
    p = model.predict_proba(B)[:, 1]
    print(f"{name:22s} test error {np.mean((p >= 0.5) != y_test):.3f}   AUC {roc_auc_score(y_test, p):.3f}")
    return model

report(LogisticRegression(max_iter=1000), "logistic regression", standardise=True)
full = report(DecisionTreeClassifier(random_state=0), "tree (full)")
print("   full tree:", full.get_n_leaves(), "leaves, depth", full.get_depth())

### Pruning by cross-validation

In [ ]:
alphas = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_train, y_train).ccp_alphas
search = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas},
                      cv=KFold(10, shuffle=True, random_state=0)).fit(X_train, y_train)
pruned = report(search.best_estimator_, "tree (pruned by CV)")
print("   pruned tree:", pruned.get_n_leaves(), "leaves, depth", pruned.get_depth())
print(export_text(pruned, feature_names=names, decimals=1))
plt.figure(figsize=(8, 4)); plot_tree(pruned, feature_names=names, class_names=["healthy", "faulty"], filled=True); plt.show()

### Many trees: bagging, random forest, boosting

In [ ]:
report(BaggingClassifier(DecisionTreeClassifier(), n_estimators=300, random_state=0), "bagging")
rf = report(RandomForestClassifier(n_estimators=300, max_features="sqrt", random_state=0, oob_score=True), "random forest")
print("   random forest out-of-bag error:", round(1 - rf.oob_score_, 3))
report(GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=2, random_state=0), "boosting")

order = np.argsort(rf.feature_importances_)
plt.barh(np.array(names)[order], 100 * rf.feature_importances_[order])
plt.xlabel("importance (%)"); plt.title("random forest variable importance"); plt.show()

In [ ]:
sizes = [1, 5, 10, 25, 50, 100, 200, 300]
rf_err, bag_err = [], []
for nt in sizes:
    rf_err.append(np.mean(RandomForestClassifier(n_estimators=nt, max_features="sqrt", random_state=0).fit(X_train, y_train).predict(X_test) != y_test))
    bag_err.append(np.mean(BaggingClassifier(DecisionTreeClassifier(), n_estimators=nt, random_state=0).fit(X_train, y_train).predict(X_test) != y_test))
plt.semilogx(sizes, rf_err, "o-", label="random forest"); plt.semilogx(sizes, bag_err, "s-", label="bagging")
plt.xlabel("number of trees"); plt.ylabel("test error"); plt.legend(); plt.show()

**Questions**
4. Compare the pruned tree with the hidden rules in `plant`. Which rule did it find? Which did it miss, and why might that be?
5. Why does logistic regression do worst here?
6. Humidity plays no part in the faults. Why does it still get some importance? Drop it (use columns 0–4 only) and compare the random forest's test error.
7. The out-of-bag error is an estimate of the test error. How close is it here? Why might they differ with only 500 training machines?
8. **Your group project:** fit a random forest to your data, compare its 10-fold CV error with your baseline, and plot the importance.

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `h`, `temp`, `y`, `X2`, `gam`, `cv_mse`, `demand`, `plant`, `X_train`, `y_train`, `X_test`, `y_test` and `names`
from above, so run the whole notebook first.

### Task 1: Trees for a regression problem

Trees also predict numbers: each leaf predicts the **average** of its training points.

(a) Fit `DecisionTreeRegressor` to the electricity demand with both inputs (`X2`: hour and temperature). Choose `max_depth` from 1 to 10
by 10-fold CV with `cv_mse`. Give the CV error of the best depth.

(b) Fit `RandomForestRegressor(n_estimators=300, random_state=0)` and give its CV error. Compare both with the GAM of Part 1.

(c) Plot the predictions of the best tree, the forest and the GAM against the hour, at a fixed temperature of 24 °C, together with the
truth `demand(g, 24)`. Why does the tree look like a staircase? Which model would you give to the building manager?

In [ ]:
# Task 1: your code here

### Task 2: Tuning a random forest with the out-of-bag error

A random forest has its own free estimate of the test error: the **out-of-bag (OOB)** error. Use it to tune the forest without
touching the test machines.

(a) For `max_features` = 1, 2, 3, 4, 5, 6, fit `RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0)` on the
training machines and record the OOB error (`1 - oob_score_`). Plot it.

(b) Which value is best? What is `max_features = 6` the same as?

(c) Fit the best forest and give its **test** error. Then refit it on columns 0–4 only (no humidity) and compare.

In [ ]:
# Task 2: your code here

### Task 3: Can boosting overfit?

Boosting adds small trees one at a time, each fixing the errors of the ones before. Unlike a random forest, too many trees **can** overfit.
We watch two scores on validation machines: the error rate, and the **cross-entropy** of Lecture 3 (`log_loss` in scikit-learn),
which also checks whether the predicted *probabilities* are sensible.

(a) Split the 500 training machines into 350 for fitting and 150 for validation.

(b) Fit `GradientBoostingClassifier(n_estimators=1000, max_depth=2, random_state=0)` with learning rates 0.5, 0.1 and 0.02.
For each, use `staged_predict` and `staged_predict_proba` to get the validation error rate and cross-entropy after 1, 2, ..., 1000 trees.
Plot the cross-entropy curves on one graph and the error-rate curves on another (log scale for the number of trees).

(c) For each learning rate, how many trees give the lowest validation cross-entropy? The error rate hardly changes after many trees,
but the cross-entropy rises. What is happening to the predicted probabilities? Why would that matter for choosing a threshold, as in Lecture 3?

(d) Explain the link between the learning rate and the number of trees.

In [ ]:
# Task 3: your code here